# CarPlate Colab Pipeline

This notebook combines dependency installation, data preparation, custom model definition, training, and model comparison into one Colab-ready workflow.

Assumption: dataset files are already downloaded in the runtime (for example under `data/images` and `data/labels`).

In [ ]:
!pip install -q ultralytics torch torchvision pillow pyyaml numpy

In [ ]:
import os
import random
import shutil
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from ultralytics import YOLO

SUPPORTED_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def split_data(image_dir, label_dir, output_dir, split_ratio=0.8, seed=42):
    random.seed(seed)

    for split in ("train", "val"):
        os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)

    images = [f for f in os.listdir(image_dir) if f.lower().endswith(SUPPORTED_IMAGE_EXTENSIONS)]
    if not images:
        raise ValueError(f"No images found in {image_dir}")

    random.shuffle(images)
    train_count = int(len(images) * split_ratio)
    splits = {"train": images[:train_count], "val": images[train_count:]}

    for split_name, files in splits.items():
        for image_name in files:
            image_src = os.path.join(image_dir, image_name)
            image_dst = os.path.join(output_dir, "images", split_name, image_name)
            shutil.copy2(image_src, image_dst)

            label_name = os.path.splitext(image_name)[0] + ".txt"
            label_src = os.path.join(label_dir, label_name)
            if os.path.exists(label_src):
                label_dst = os.path.join(output_dir, "labels", split_name, label_name)
                shutil.copy2(label_src, label_dst)

    yaml_data = {
        "path": os.path.abspath(output_dir),
        "train": "images/train",
        "val": "images/val",
        "nc": 1,
        "names": ["car_plate"],
    }
    yaml_path = os.path.join(output_dir, "data.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(yaml_data, f, sort_keys=False)

    print(f"Data split complete: train={len(splits['train'])}, val={len(splits['val'])}")
    print(f"YOLO data config written to {yaml_path}")


In [ ]:
def autopad(k, p=None, d=1):
    if d > 1:
        k = d * (k - 1) + 1 if isinstance(k, int) else [d * (x - 1) + 1 for x in k]
    if p is None:
        p = k // 2 if isinstance(k, int) else [x // 2 for x in k]
    return p


class Conv(nn.Module):
    """Conv + BN + SiLU."""

    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Bottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, g=1, k=(3, 3), e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, k[0], 1)
        self.cv2 = Conv(c_, c2, k[1], 1, g=g)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        y = self.cv2(self.cv1(x))
        return x + y if self.add else y


class C2f(nn.Module):
    """YOLOv8-style C2f block."""

    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5, bottleneck_expansion=1.0):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1, 1)
        self.m = nn.ModuleList(
            Bottleneck(self.c, self.c, shortcut=shortcut, g=g, k=(3, 3), e=bottleneck_expansion) for _ in range(n)
        )

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))


class SPPF(nn.Module):
    """Spatial Pyramid Pooling - Fast."""

    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c_ * 4, c2, 1, 1)
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        return self.cv2(torch.cat((x, y1, y2, y3), 1))


class DecoupledHead(nn.Module):
    """Decoupled cls/box prediction head for a single scale."""

    def __init__(self, in_channels, num_classes=1):
        super().__init__()
        self.cls_conv = Conv(in_channels, in_channels, 3, 1)
        self.cls_pred = nn.Conv2d(in_channels, num_classes, 1)

        self.box_conv = Conv(in_channels, in_channels, 3, 1)
        self.box_pred = nn.Conv2d(in_channels, 4, 1)

    def forward(self, x):
        cls_out = self.cls_pred(self.cls_conv(x))
        box_out = self.box_pred(self.box_conv(x))
        return cls_out, box_out


class ImprovedCustomYOLO(nn.Module):
    """Custom YOLO architecture with C2f backbone, SPPF, top-down FPN and multi-scale decoupled heads."""

    def __init__(self, num_classes=1):
        super().__init__()
        self.stem = Conv(3, 16, 3, 2)
        self.layer1 = nn.Sequential(Conv(16, 32, 3, 2), C2f(32, 32, n=1, shortcut=True))
        self.layer2 = nn.Sequential(Conv(32, 64, 3, 2), C2f(64, 64, n=2, shortcut=True))
        self.layer3 = nn.Sequential(Conv(64, 128, 3, 2), C2f(128, 128, n=2, shortcut=True))
        self.layer4 = nn.Sequential(Conv(128, 256, 3, 2), C2f(256, 256, n=1, shortcut=True))
        self.sppf = SPPF(256, 256, k=5)

        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.neck_c2f_1 = C2f(256 + 128, 128, n=1, shortcut=False)
        self.neck_c2f_2 = C2f(128 + 64, 64, n=1, shortcut=False)

        self.head_small = DecoupledHead(64, num_classes)
        self.head_medium = DecoupledHead(128, num_classes)
        self.head_large = DecoupledHead(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)

        p3 = self.layer2(x)
        p4 = self.layer3(p3)
        p5 = self.sppf(self.layer4(p4))

        p4_fused = self.neck_c2f_1(torch.cat([self.upsample(p5), p4], dim=1))
        p3_fused = self.neck_c2f_2(torch.cat([self.upsample(p4_fused), p3], dim=1))

        out_small = self.head_small(p3_fused)
        out_medium = self.head_medium(p4_fused)
        out_large = self.head_large(p5)

        return out_small, out_medium, out_large


In [ ]:
def train_official_yolo(data_yaml="dataset/data.yaml", weights="yolov8n.pt", epochs=50, imgsz=640, batch=16):
    model = YOLO(weights)
    model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        project="runs/official_yolo",
        name="car_plate_model",
    )
    print("Training complete. Best weights: runs/official_yolo/car_plate_model/weights/best.pt")


def train_custom_model(
    epochs=50,
    batch_size=8,
    lr=1e-3,
    save_path="runs/custom_yolo/custom_yolo_best.pth",
    batches_per_epoch=10,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ImprovedCustomYOLO(num_classes=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    cls_loss_fn = nn.BCEWithLogitsLoss()
    box_loss_fn = nn.MSELoss()

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    best_loss = float("inf")
    model.train()
    print(f"Training custom model on {device}")
    print("Warning: this script uses placeholder synthetic data/targets for structural training only.")

    for epoch in range(epochs):
        epoch_loss = 0.0

        for _ in range(batches_per_epoch):
            images = torch.randn(batch_size, 3, 640, 640, device=device)

            optimizer.zero_grad()
            (cls_s, box_s), (cls_m, box_m), (cls_l, box_l) = model(images)

            target_cls_s = torch.zeros_like(cls_s)
            target_box_s = torch.zeros_like(box_s)
            target_cls_m = torch.zeros_like(cls_m)
            target_box_m = torch.zeros_like(box_m)
            target_cls_l = torch.zeros_like(cls_l)
            target_box_l = torch.zeros_like(box_l)

            loss = (
                cls_loss_fn(cls_s, target_cls_s)
                + box_loss_fn(box_s, target_box_s)
                + cls_loss_fn(cls_m, target_cls_m)
                + box_loss_fn(box_m, target_box_m)
                + cls_loss_fn(cls_l, target_cls_l)
                + box_loss_fn(box_l, target_box_l)
            )
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / batches_per_epoch
        print(f"Epoch [{epoch + 1}/{epochs}] loss={avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), save_path)
            print(f"Saved best custom checkpoint to {save_path}")


In [ ]:
def get_model_size_mb(model_path):
    return os.path.getsize(model_path) / (1024 * 1024)


def benchmark_official(model, runs=50, img_size=640):
    image = np.random.randint(0, 255, (img_size, img_size, 3), dtype=np.uint8)
    start = time.perf_counter()
    for _ in range(runs):
        model.predict(source=image, verbose=False)
    avg_seconds = (time.perf_counter() - start) / runs
    return avg_seconds


def benchmark_custom(model, device, runs=50, img_size=640):
    x = torch.randn(1, 3, img_size, img_size, device=device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(runs):
            model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    avg_seconds = (time.perf_counter() - start) / runs
    return avg_seconds


def compare_models(official_weights, custom_weights, runs=50, img_size=640):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    official_model = YOLO(official_weights)
    custom_model = ImprovedCustomYOLO(num_classes=1).to(device)
    custom_model.load_state_dict(torch.load(custom_weights, map_location=device))
    custom_model.eval()

    official_size = get_model_size_mb(official_weights)
    custom_size = get_model_size_mb(custom_weights)

    official_time = benchmark_official(official_model, runs=runs, img_size=img_size)
    custom_time = benchmark_custom(custom_model, device=device, runs=runs, img_size=img_size)

    print("\n" + "=" * 64)
    print("YOLOv8 Official vs ImprovedCustomYOLO")
    print("=" * 64)
    print(f"{'Metric':<24}{'Official YOLOv8':<20}{'ImprovedCustomYOLO':<20}")
    print("-" * 64)
    print(f"{'Model Size (MB)':<24}{official_size:<20.2f}{custom_size:<20.2f}")
    print(f"{'Inference Time (ms)':<24}{official_time * 1000:<20.2f}{custom_time * 1000:<20.2f}")
    print(f"{'Estimated FPS':<24}{(1 / official_time):<20.2f}{(1 / custom_time):<20.2f}")
    print("=" * 64)


In [ ]:
# Update these paths as needed in Colab
image_dir = "data/images"
label_dir = "data/labels"
dataset_dir = "dataset"
data_yaml = os.path.join(dataset_dir, "data.yaml")

# Prepare split/data.yaml only if needed
if not os.path.exists(data_yaml):
    split_data(image_dir=image_dir, label_dir=label_dir, output_dir=dataset_dir)

# Train official YOLOv8 model
train_official_yolo(data_yaml=data_yaml, weights="yolov8n.pt", epochs=50, imgsz=640, batch=16)

# Train custom model
train_custom_model(epochs=50, batch_size=8, lr=1e-3, save_path="runs/custom_yolo/custom_yolo_best.pth")

# Compare both trained models
compare_models(
    official_weights="runs/official_yolo/car_plate_model/weights/best.pt",
    custom_weights="runs/custom_yolo/custom_yolo_best.pth",
    runs=50,
    img_size=640,
)
